# YOLO-World FPS check

Previous metrics reused.

In [ ]:
# Cell 1-A Setup

import sys
!{sys.executable} -m pip install "ultralytics>=8.1.0"
!{sys.executable} -m pip install lvis

import glob
for p in glob.glob("/usr/local/lib/python*/dist-packages/lvis/eval.py"):
    txt = open(p).read().replace("np.float", "float")
    open(p, "w").write(txt)
    print("patched:", p)

print("Install complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.2 MB/s eta 0:00:00
  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.3 MB/s eta 0:00:00
Using cached ultralytics_thop-2.0.19-py3-none-any.whl (28 kB)
  Using cached lvis-0.5.3-py3-none-any.whl.metadata (856 bytes)
Using cached lvis-0.5.3-py3-none-any.whl (14 kB)
patched: /usr/local/lib/python3.12/dist-packages/lvis/eval.py
Install complete


In [ ]:
# Cell 1-B import

from ultralytics import YOLOWorld
print("YOLOWorld import OK")

WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOWorld import OK


In [ ]:
# Cell 2 - LVIS annotation

from google.colab import files
import shutil, json
from pathlib import Path

Path("/content/datasets/lvis/annotations").mkdir(parents=True, exist_ok=True)

# print("Please upload lvis_v1_minival.json:")
# files.upload()
# Manual upload

shutil.copy(
    "/content/lvis_v1_minival.json",
    "/content/datasets/lvis/annotations/lvis_v1_minival.json"
)

with open("/content/datasets/lvis/annotations/lvis_v1_minival.json") as f:
    ann = json.load(f)
print(f"✓ images: {len(ann['images'])}, categories: {len(ann['categories'])}")

✓ images: 4809, categories: 1203


## Own Code
Sets LVIS category names in YOLO-World and measures FPS with repeated prediction under the same local runtime.

In [ ]:
# Cell 3 - FPS

import sys, time, json, statistics, torch
import numpy as np

from ultralytics import YOLOWorld

# Load LVIS categories
with open("/content/datasets/lvis/annotations/lvis_v1_minival.json") as f:
    ann = json.load(f)
cats = [c["name"] for c in sorted(ann["categories"], key=lambda x: x["id"])]
print(f"LVIS categories: {len(cats)}")

# Model and categories
model = YOLOWorld("yolov8s-worldv2.pt")  # Auto-download
model.set_classes(cats)

dummy = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

# Warmup
print("Warming up...")
for _ in range(10):
    model.predict(dummy, verbose=False, imgsz=640)

# Wall-clock FPS
N = 200
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N):
    model.predict(dummy, verbose=False, imgsz=640)
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
fps_wall = N / elapsed
ms_wall = elapsed * 1000 / N

# Ultralytics reported speed
speed_log = {"preprocess": [], "inference": [], "postprocess": []}
for _ in range(100):
    r = model.predict(dummy, verbose=False, imgsz=640)
    for k in speed_log:
        speed_log[k].append(r[0].speed[k])

avg = {k: statistics.mean(v) for k, v in speed_log.items()}
total_ms = sum(avg.values())

print(f"\n===== YOLO-World-v2-S PyTorch FPS (batch=1, 640×640) =====")
print(f"[Wall-clock]     {fps_wall:.1f} FPS  ({ms_wall:.2f} ms/img)")
print(f"[Inference only] {1000/avg['inference']:.1f} FPS  ({avg['inference']:.2f} ms)")
print(f"[End-to-end]     {1000/total_ms:.1f} FPS")
print(f"  pre={avg['preprocess']:.2f}  inf={avg['inference']:.2f}  post={avg['postprocess']:.2f} ms")
print(f"\nPaper TensorRT T4: 216.4 FPS  (different setting)")

result = {
    "wall_clock_fps": round(fps_wall, 1),
    "inference_only_fps": round(1000/avg["inference"], 1),
    "end_to_end_fps": round(1000/total_ms, 1),
    "speed_ms": {k: round(v, 2) for k, v in avg.items()},
}
with open("/content/fps_yoloworld.json", "w") as f:
    json.dump(result, f, indent=2)
print("saved fps_yoloworld.json")

LVIS categories: 1203
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 37 packages in 562ms
Prepared 2 packages in 2.46s
Installed 2 packages in 1ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@81ff68ed7ffcac3b40484c914f104f816757308d)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:00<00:00, 364MiB/s]


Warming up...

===== YOLO-World-v2-S PyTorch FPS (batch=1, 640×640) =====
[Wall-clock]     73.7 FPS  (13.56 ms/img)
[Inference only] 86.6 FPS  (11.54 ms)
[End-to-end]     73.9 FPS
  pre=1.41  inf=11.54  post=0.58 ms

Paper TensorRT T4: 216.4 FPS  (different setting)
saved fps_yoloworld.json
